# Patient Demographics for ACM Publication

**Purpose**: Generate demographic tables for the ACM publication based on the new schema `pain_narratives_acm_202512`.

**Objectives**:
1. Generate Table 1: Demographics for all patients (N=152)
2. Generate Table 2: Demographics for patients with narratives evaluated by experts

**Key Requirements** (from reviewer feedback):
- Show only actual numbers (no percentages)
- Two separate tables: all patients vs. expert-evaluated subset
- Categories: Demographics, Education, Employment Status, Pain Characteristics, Primary Pain Diagnosis, Narrative Characteristics

**Data Source**: PostgreSQL schema `pain_narratives_acm_202512`

**Created**: 2026-01-11

## 1. Environment Setup

In [1]:
# Standard library imports
import sys
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sqlmodel import Session

# Add src to path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root / 'src'))

# Import database manager
from pain_narratives.core.database import DatabaseManager

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.1f}'.format)

print("✅ Libraries imported successfully")
print(f"📁 Working directory: {notebook_dir}")
print(f"🕐 Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries imported successfully
📁 Working directory: /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks
🕐 Analysis run: 2026-05-26 15:10:51


## 2. Load Data from New Schema

In [2]:
print("="*80)
print("LOADING DATA FROM ai_narratives_original SCHEMA")
print("="*80)

# Initialize database manager
db = DatabaseManager()

# Load all patient data
query_all = """
SELECT 
    participant_id,
    narrative_text,
    narrative_hash,
    word_count,
    age,
    gender,
    marital_status,
    education_level,
    country_residence,
    country_birth,
    employment_status,
    years_with_pain,
    years_since_diagnosis,
    pain_cause_primary,
    pain_location_zones
FROM ai_narratives_original.real_patient_demographics
ORDER BY participant_id
"""

with Session(db.engine) as session:
    df_all_patients = pd.read_sql(text(query_all), session.connection())

print(f"\n✅ Loaded {len(df_all_patients)} patient records")
print(f"\nColumns: {list(df_all_patients.columns)}")
print(f"\nSample data:")
df_all_patients.head(3)

LOADING DATA FROM ai_narratives_original SCHEMA



✅ Loaded 152 patient records

Columns: ['participant_id', 'narrative_text', 'narrative_hash', 'word_count', 'age', 'gender', 'marital_status', 'education_level', 'country_residence', 'country_birth', 'employment_status', 'years_with_pain', 'years_since_diagnosis', 'pain_cause_primary', 'pain_location_zones']

Sample data:


,participant_id,narrative_text,narrative_hash,word_count,age,gender,marital_status,education_level,country_residence,country_birth,employment_status,years_with_pain,years_since_diagnosis,pain_cause_primary,pain_location_zones
0,1,Lo recuerdo desde la infancia.\nEs un dolor ge...,8eb52cdf593f14d8c0af0b17d265e47ec85f6d5173604f...,63,56,Femenino,Casado/a,Estudios de primaria,España,España,Baja laboral transitoria,23 años,20 años,Dolor crónico primario,"Cabeza,Hombro,Brazo,Tórax,Abdomen,Cervical,Lum..."
1,2,Mis dolores empezaron en alguna zona en concre...,aa2889eba8a5b6c6346ea5ad694732885ecadb803e4047...,92,56,Femenino,Casado/a,"Estudio secundarios (ESO, bachillerato, ciclos...",España,España,Pensión de incapacidad/invalidez permanente,12 años,5 años,Dolor crónico primario,"Cabeza,Hombro,Brazo,Tórax,Abdomen,Cervical,Lum..."
2,3,Mi vida desde los 48 años ya no es la misma yo...,a74067761ba5d7286ca8c81e00f9a84dca049f58686ea9...,39,52,Femenino,Separado/a,Estudios de primaria,España,España,Baja laboral transitoria,3 años,2 años,Dolor crónico primario,"Cabeza,Hombro,Brazo,Tórax,Abdomen,Cervical,Lum..."


## 3. Identify Narratives Used in Expert Evaluation

In [3]:
print("="*80)
print("IDENTIFYING EXPERT-EVALUATED NARRATIVES")
print("="*80)

# Load expert evaluation data to identify which narratives were evaluated
query_expert = """
SELECT DISTINCT 
    participant_id,
    narrative_hash
FROM ai_narratives_original.expert_dimension_evaluation
WHERE participant_id IS NOT NULL
"""

with Session(db.engine) as session:
    df_expert_narratives = pd.read_sql(text(query_expert), session.connection())

print(f"\n✅ Found {len(df_expert_narratives)} unique narratives evaluated by experts")

# Filter all patients to get only expert-evaluated subset
df_expert_patients = df_all_patients[
    df_all_patients['participant_id'].isin(df_expert_narratives['participant_id'])
].copy()

print(f"✅ Expert-evaluated patient subset: {len(df_expert_patients)} patients")
print(f"\nParticipant IDs in expert evaluation: {sorted(df_expert_patients['participant_id'].tolist())}")

IDENTIFYING EXPERT-EVALUATED NARRATIVES

✅ Found 22 unique narratives evaluated by experts
✅ Expert-evaluated patient subset: 22 patients

Participant IDs in expert evaluation: [2, 3, 4, 11, 15, 16, 19, 20, 25, 32, 37, 38, 46, 48, 50, 51, 52, 57, 58, 63, 69, 78]


## 4. Helper Functions for Table Generation

In [4]:
def parse_spanish_duration(text):
    """
    Parse Spanish duration strings to numeric years.
    Examples: "23 años" -> 23.0, "Menos de un año" -> 0.5
    """
    import re
    
    if pd.isna(text) or text is None:
        return None
    
    text = str(text).strip().lower()
    
    # Handle "menos de un año" (less than a year)
    if 'menos de un año' in text or 'menos de 1 año' in text:
        return 0.5
    
    # Try to extract number followed by "año" or "años"
    match = re.search(r'(\d+)\s*años?', text)
    if match:
        return float(match.group(1))
    
    # Try to extract just a number (if already numeric)
    try:
        return float(text)
    except (ValueError, TypeError):
        pass
    
    return None


def generate_demographics_table(df, table_name="All Patients"):
    """
    Generate demographic table with only actual numbers (no percentages).
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with patient data
    table_name : str
        Name for the table (e.g., "All Patients" or "Expert-Evaluated Subset")
    
    Returns:
    --------
    pd.DataFrame
        Formatted demographic table
    """
    table_data = []
    
    # Sample Size
    table_data.append(['Sample Size', len(df)])
    table_data.append(['', ''])
    
    # Demographics Section
    table_data.append(['Demographics', ''])
    
    # Age
    age_data = df['age'].dropna()
    if len(age_data) > 0:
        table_data.append(['  Age, years — mean (SD)', f'{age_data.mean():.1f} ({age_data.std():.1f})'])
        table_data.append(['  Age range', f'[{age_data.min():.0f}, {age_data.max():.0f}]'])
    
    # Gender - ONLY ACTUAL NUMBERS, NO PERCENTAGES
    gender_counts = df['gender'].value_counts()
    for gender, count in gender_counts.items():
        table_data.append([f'  {gender}', str(count)])
    
    # Education
    table_data.append(['', ''])
    table_data.append(['Education', ''])
    edu_counts = df['education_level'].value_counts()
    for level, count in edu_counts.items():
        table_data.append([f'  {level}', str(count)])
    
    # Employment Status (top 5)
    table_data.append(['', ''])
    table_data.append(['Employment Status', ''])
    employ_counts = df['employment_status'].value_counts().head(5)
    for status, count in employ_counts.items():
        table_data.append([f'  {status}', str(count)])
    
    # Pain Characteristics
    table_data.append(['', ''])
    table_data.append(['Pain Characteristics', ''])
    
    # Parse duration fields
    df_temp = df.copy()
    df_temp['years_with_pain_numeric'] = df_temp['years_with_pain'].apply(parse_spanish_duration)
    df_temp['years_since_diagnosis_numeric'] = df_temp['years_since_diagnosis'].apply(parse_spanish_duration)
    
    pain_duration = df_temp['years_with_pain_numeric'].dropna()
    if len(pain_duration) > 0:
        table_data.append(['  Years with pain — mean (SD)', f'{pain_duration.mean():.1f} ({pain_duration.std():.1f})'])
    
    years_diagnosed = df_temp['years_since_diagnosis_numeric'].dropna()
    if len(years_diagnosed) > 0:
        table_data.append(['  Years since diagnosis — mean (SD)', f'{years_diagnosed.mean():.1f} ({years_diagnosed.std():.1f})'])
    
    # Primary Pain Diagnosis (top 5)
    table_data.append(['', ''])
    table_data.append(['Primary Pain Diagnosis', ''])
    cause_counts = df['pain_cause_primary'].value_counts().head(5)
    for cause, count in cause_counts.items():
        table_data.append([f'  {cause}', str(count)])
    
    # Narrative Characteristics
    table_data.append(['', ''])
    table_data.append(['Narrative Characteristics', ''])
    word_counts = df['word_count'].dropna()
    if len(word_counts) > 0:
        table_data.append(['  Word count — mean (SD)', f'{word_counts.mean():.0f} ({word_counts.std():.0f})'])
        table_data.append(['  Word count range', f'[{word_counts.min():.0f}, {word_counts.max():.0f}]'])
    
    # Create DataFrame
    df_table = pd.DataFrame(table_data, columns=['Characteristic', 'Value'])
    
    return df_table

print("✅ Helper functions defined")

✅ Helper functions defined


## 5. Generate Table 1: All Patients (N=152)

In [5]:
print("="*80)
print("TABLE 1: ALL PATIENTS DEMOGRAPHICS")
print("="*80)

table1_all_patients = generate_demographics_table(df_all_patients, "All Patients")

print("\n")
print(table1_all_patients.to_string(index=False))

# Export table
output_dir = notebook_dir / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / '02_table1_all_patients_demographics.csv'
table1_all_patients.to_csv(output_file, index=False)
print(f"\n✅ Exported: {output_file}")

TABLE 1: ALL PATIENTS DEMOGRAPHICS


                                                     Characteristic       Value
                                                        Sample Size         152
                                                                               
                                                       Demographics            
                                             Age, years — mean (SD) 49.8 (10.5)
                                                          Age range    [21, 70]
                                                           Femenino         137
                                                          Masculino          13
                                                         No binario           1
                                            Click to write Choice 4           1
                                                                               
                                                          Education            
   

## 6. Generate Table 2: Expert-Evaluated Patients

In [6]:
print("="*80)
print("TABLE 2: EXPERT-EVALUATED PATIENTS DEMOGRAPHICS")
print("="*80)

table2_expert_patients = generate_demographics_table(df_expert_patients, "Expert-Evaluated Subset")

print("\n")
print(table2_expert_patients.to_string(index=False))

# Export table
output_file = output_dir / '02_table2_expert_evaluated_patients_demographics.csv'
table2_expert_patients.to_csv(output_file, index=False)
print(f"\n✅ Exported: {output_file}")

TABLE 2: EXPERT-EVALUATED PATIENTS DEMOGRAPHICS


                                                     Characteristic       Value
                                                        Sample Size          22
                                                                               
                                                       Demographics            
                                             Age, years — mean (SD) 49.6 (11.3)
                                                          Age range    [31, 69]
                                                           Femenino          19
                                                          Masculino           2
                                                         No binario           1
                                                                               
                                                          Education            
  Estudio secundarios (ESO, bachillerato, ciclos formativos, otros)   

## 7. Side-by-Side Comparison

In [7]:
print("="*80)
print("SIDE-BY-SIDE COMPARISON")
print("="*80)

# Merge tables for comparison
comparison = table1_all_patients.merge(
    table2_expert_patients,
    on='Characteristic',
    how='outer',
    suffixes=('_All', '_Expert')
)

comparison.columns = ['Characteristic', 'All Patients (N=152)', 'Expert-Evaluated Subset']

print("\n")
print(comparison.to_string(index=False))

# Export comparison
output_file = output_dir / '02_table_comparison_all_vs_expert.csv'
comparison.to_csv(output_file, index=False)
print(f"\n✅ Exported: {output_file}")

SIDE-BY-SIDE COMPARISON


                                                     Characteristic All Patients (N=152) Expert-Evaluated Subset
                                                                                                                
                                                                                                                
                                                                                                                
                                                                                                                
                                                                                                                
                                                                                                                
                                                                                                                
                                                                      

## 8. Summary Statistics

In [8]:
print("\n" + "="*80)
print("✅ PATIENT DEMOGRAPHICS ANALYSIS COMPLETE")
print("="*80)

print(f"\n📊 Summary:")
print(f"   Total patients: {len(df_all_patients)}")
print(f"   Expert-evaluated patients: {len(df_expert_patients)}")
print(f"   Percentage evaluated: {len(df_expert_patients)/len(df_all_patients)*100:.1f}%")

print(f"\n📁 Outputs saved to: {output_dir}")
print(f"   - table1_all_patients_demographics.csv")
print(f"   - table2_expert_evaluated_patients_demographics.csv")
print(f"   - table_comparison_all_vs_expert.csv")

print(f"\n💡 Note: Tables show ONLY actual numbers (no percentages) as requested in reviewer feedback")


✅ PATIENT DEMOGRAPHICS ANALYSIS COMPLETE

📊 Summary:
   Total patients: 152
   Expert-evaluated patients: 22
   Percentage evaluated: 14.5%

📁 Outputs saved to: /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks/outputs
   - table1_all_patients_demographics.csv
   - table2_expert_evaluated_patients_demographics.csv
   - table_comparison_all_vs_expert.csv

💡 Note: Tables show ONLY actual numbers (no percentages) as requested in reviewer feedback
